In [ ]:
#| default_exp features.wps_background

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()
        

In [ ]:
#| export
def _parse_array(s):
    if not isinstance(s, str) or not s.startswith('['): 
        return []
    clean = s.replace('[', '').replace(']', '').replace(chr(10), '').replace(chr(13), '')
    try:
        return [float(x) for x in clean.split()]
    except:
        return []

class WPSBackgroundEvaluator(FeatureEvaluator):
    """Extracts periodicity distances for nucleosomes."""
    
    name = "WPSBackground"
    source_file = ".WPS_background.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "group_id" in cols:
                metrics = ["nrl_bp", "nrl_deviation_bp", "periodicity_score", "adjusted_score", "fragment_ratio"]
                for _, row in df.iterrows():
                    gi = str(row["group_id"]).replace(" ", "_")
                    for m in metrics:
                        if m in cols and pd.notna(row[m]):
                            extracted[f"{gi}_{m}"] = float(row[m])
    
            return extracted
        except Exception as e:
            log.warning("wps_background_extraction_failed", error=str(e))
            return {}


In [ ]:
#| test
evaluator = WPSBackgroundEvaluator()
df = pd.DataFrame([{"group_id": "Chr1", "nrl_bp": 150.0}])
res = evaluator.extract(df)
assert res["Chr1_nrl_bp"] == 150.0
